# ViRecognAgent — Structured Vietnamese Document Extraction

**Goal:** Combine PaddleOCR PP-StructureV3 (layout analysis) with Gemma 4 E4B (vision-language) to extract structured data (forms, tables) from Vietnamese handwritten/printed documents.

## Pipeline
```
Image
  → PP-StructureV3  (detect text regions, tables, figures with bounding boxes)
  → PaddleOCR OCR  (read text in each region → JSON)
  → Gemma 4 E4B    (vision + OCR JSON → structured output: fields, tables)
  → Parse + validate JSON output
```

## Why this combination?
- **PP-StructureV3** excels at *layout detection* — finds where text/tables/figures are
- **PaddleOCR** reads the text reliably (especially with `lang='vi'`)
- **Gemma 4 E4B** understands Vietnamese context, corrects OCR errors, and maps fields to schema

**Runtime:** T4 GPU. Gemma 4 E4B at 4-bit quantization fits in ~8GB VRAM.

### Secrets expected in Colab (🔑 Secrets sidebar)
- `HF_TOKEN` — HuggingFace read token (for Gemma 4 gated model)
- `GITHUB_TOKEN` — optional, for pushing fixes from Colab

## 1. Environment & GPU check

In [ ]:
!nvidia-smi | head -n 20
import sys, platform
print(f"Python: {sys.version.split()[0]}  |  Platform: {platform.platform()}")

## 2. Secrets + Google Drive cache

In [ ]:
import os, pathlib

def _get_colab_secret(name, retries=2, sleep=1.0):
    import time
    from google.colab import userdata
    last = None
    for _ in range(retries + 1):
        try:
            return userdata.get(name)
        except Exception as e:
            last = e
            time.sleep(sleep)
    raise last

try:
    HF_TOKEN = _get_colab_secret('HF_TOKEN')
    try:
        GITHUB_TOKEN = _get_colab_secret('GITHUB_TOKEN')
    except Exception:
        GITHUB_TOKEN = None
except ImportError:
    HF_TOKEN = os.environ.get('HF_TOKEN')
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')

assert HF_TOKEN, "HF_TOKEN not found — add it to Colab Secrets (key icon) and re-run."
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
print('HF_TOKEN:', 'set ok' if HF_TOKEN else 'MISSING')
print('GITHUB_TOKEN:', 'set ok' if GITHUB_TOKEN else 'not set (ok)')

# Mount Google Drive for persistent model/dataset caching
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CACHE = '/content/drive/MyDrive/virecognagent_cache'
    pathlib.Path(DRIVE_CACHE).mkdir(parents=True, exist_ok=True)

    HF_CACHE = f'{DRIVE_CACHE}/hf_datasets'
    pathlib.Path(HF_CACHE).mkdir(exist_ok=True)
    os.environ['HF_DATASETS_CACHE'] = HF_CACHE

    for _subdir in ('.paddleocr', '.paddlex'):
        _local = os.path.expanduser(f'~/{_subdir}')
        _drive = f'{DRIVE_CACHE}/{_subdir}'
        pathlib.Path(_drive).mkdir(exist_ok=True)
        if not os.path.exists(_local):
            os.symlink(_drive, _local)
        elif not os.path.islink(_local):
            import shutil
            shutil.copytree(_local, _drive, dirs_exist_ok=True)
            shutil.rmtree(_local)
            os.symlink(_drive, _local)

    # Gemma 4 model cache on Drive (~4GB 4-bit quant)
    HF_HOME_DRIVE = f'{DRIVE_CACHE}/hf_home'
    pathlib.Path(HF_HOME_DRIVE).mkdir(exist_ok=True)
    os.environ['HF_HOME'] = HF_HOME_DRIVE

    print(f'Drive mounted. Caches:')
    print(f'  HF datasets : {HF_CACHE}')
    print(f'  HF home     : {HF_HOME_DRIVE}')
    print(f'  PaddleOCR   : {DRIVE_CACHE}/.paddleocr')
except Exception as e:
    print(f'Drive not available ({e}) — using local cache only')

## 3. Install dependencies

- `paddlepaddle` CPU-only (from official index — avoids CUDA wheel hell)
- `paddleocr>=3.0` — PP-StructureV3 + OCR API
- `transformers>=4.50.0` — required for Gemma 4
- `accelerate`, `bitsandbytes` — 4-bit quantization for T4 VRAM

In [ ]:
!pip install -q 'paddlepaddle==3.0.0' -i https://www.paddlepaddle.org.cn/packages/stable/cpu/ 2>&1 | tail -3
!pip install -q 'paddleocr>=3.0' 2>&1 | tail -3
!pip install -q 'transformers>=4.50.0' accelerate bitsandbytes 'Pillow>=10' jiwer 2>&1 | tail -3

import paddle, paddleocr
import transformers
print(f'Paddle:       {paddle.__version__}  |  CUDA compiled: {paddle.is_compiled_with_cuda()}')
print(f'PaddleOCR:    {paddleocr.__version__}')
print(f'Transformers: {transformers.__version__}')

## 4. Create a synthetic Vietnamese form

We render a Vietnamese registration form (PIL) as our test image. Includes:
- Named fields (Ho ten, Ngay sinh, Dia chi, CMND)
- A 3-row medical history table

Replace `TEST_IMAGE_PATH` with a real scan to use this on actual documents.

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import matplotlib.pyplot as plt

W, H = 900, 700
img = Image.new('RGB', (W, H), 'white')
draw = ImageDraw.Draw(img)

try:
    font_lg = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 20)
    font_md = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 16)
    font_sm = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 14)
except OSError:
    font_lg = font_md = font_sm = ImageFont.load_default()

draw.text((W // 2, 20), 'PHIEU DANG KY KHAM BENH', font=font_lg, fill='black', anchor='mt')
draw.text((W // 2, 48), 'BENH VIEN DA KHOA TINH', font=font_md, fill='gray', anchor='mt')
draw.line([(30, 72), (W - 30, 72)], fill='black', width=2)

fields = [
    ('Ho va ten:', 'Nguyen Thi Hoa'),
    ('Ngay sinh:', '15/03/1985'),
    ('Gioi tinh:', 'Nu'),
    ('Dia chi:', '123 Duong Le Loi, Quan 1, TP.HCM'),
    ('So CMND/CCCD:', '012345678901'),
    ('So dien thoai:', '0901234567'),
    ('Bao hiem y te:', 'HS4010012345678'),
    ('Ly do kham:', 'Dau dau, sot 3 ngay'),
]

y = 90
for label, value in fields:
    draw.text((40, y), label, font=font_md, fill='black')
    draw.text((220, y), value, font=font_md, fill='navy')
    draw.line([(220, y + 22), (860, y + 22)], fill='lightgray', width=1)
    y += 34

y += 10
draw.text((40, y), 'TIEN SU BENH', font=font_lg, fill='black')
y += 30

cols = [40, 200, 480, 720, 860]
headers = ['STT', 'Ten benh', 'Thoi gian', 'Dieu tri']
rows_data = [
    ['1', 'Viem phe quan man', '2019-2021', 'Thuoc khang sinh'],
    ['2', 'Tang huyet ap', '2022 - nay', 'Amlodipine 5mg/ngay'],
    ['3', 'Di ung penicillin', '2015', 'Tranh dung'],
]

row_h = 30
draw.rectangle([cols[0], y, cols[-1], y + row_h], fill='#e8e8e8')
for i, hdr in enumerate(headers):
    draw.text((cols[i] + 5, y + 7), hdr, font=font_sm, fill='black')
y += row_h

for row in rows_data:
    draw.rectangle([cols[0], y, cols[-1], y + row_h], outline='gray')
    for i, cell in enumerate(row):
        draw.text((cols[i] + 5, y + 7), cell, font=font_sm, fill='black')
    for c in cols:
        draw.line([(c, y), (c, y + row_h)], fill='gray')
    y += row_h

y += 20
draw.text((600, y), 'Ngay 15 thang 04 nam 2026', font=font_sm, fill='black')
draw.text((650, y + 20), 'Chu ky benh nhan', font=font_sm, fill='gray')

TEST_IMAGE_PATH = '/tmp/viet_form_test.png'
img.save(TEST_IMAGE_PATH)
print(f'Saved test form to {TEST_IMAGE_PATH}  ({W}x{H}px)')

plt.figure(figsize=(12, 8))
plt.imshow(img)
plt.axis('off')
plt.title('Synthetic Vietnamese Registration Form')
plt.show()

## 5. PP-StructureV3 — Layout analysis + OCR

PP-StructureV3 detects layout regions (text blocks, tables, figures) and runs OCR within each region. We collect:
- `text` regions → list of `{bbox, text}` dicts
- `table` regions → HTML table string

> **Note:** PP-StructureV3 requires GPU for the layout model. If running CPU-only, we fall back to basic `PaddleOCR.predict()` which gives us text lines without layout labels.

In [ ]:
import numpy as np
from PIL import Image

pil_img = Image.open(TEST_IMAGE_PATH).convert('RGB')
img_arr = np.array(pil_img)

USE_GPU = False  # Set True if you have a GPU session with CUDA paddle

ocr_results = []

if USE_GPU:
    try:
        from paddleocr import PPStructure
        structure_engine = PPStructure(show_log=False, lang='vi')
        result = structure_engine(img_arr)
        for region in result:
            rtype = region.get('type', 'text')
            bbox = region.get('bbox', [])
            if rtype == 'table':
                html = region.get('res', {}).get('html', '')
                ocr_results.append({'type': 'table', 'bbox': bbox, 'html': html})
            else:
                for line in region.get('res', []):
                    text = line[1][0] if isinstance(line[1], (list, tuple)) else ''
                    conf = line[1][1] if isinstance(line[1], (list, tuple)) else 0.0
                    ocr_results.append({'type': 'text', 'bbox': bbox, 'text': text, 'conf': conf})
        print(f'PP-StructureV3: {len(ocr_results)} regions detected')
    except Exception as e:
        print(f'PP-StructureV3 failed: {e} — falling back to basic PaddleOCR')
        USE_GPU = False

if not USE_GPU:
    from paddleocr import PaddleOCR
    ocr = PaddleOCR(
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=True,
        lang='vi',
        device='cpu',
    )
    result = ocr.predict(input=img_arr)
    if result:
        item = result[0]
        texts = getattr(item, 'rec_texts', None) or (item.get('rec_texts') if hasattr(item, 'get') else [])
        scores = getattr(item, 'rec_scores', None) or (item.get('rec_scores') if hasattr(item, 'get') else [])
        boxes = getattr(item, 'dt_polys', None) or (item.get('dt_polys') if hasattr(item, 'get') else [])
        texts = texts or []
        scores = scores or []
        boxes = boxes or []
        for i, text in enumerate(texts):
            conf = float(scores[i]) if i < len(scores) else 0.0
            bbox = boxes[i].tolist() if i < len(boxes) else []
            ocr_results.append({'type': 'text', 'bbox': bbox, 'text': text, 'conf': conf})
    print(f'Basic PaddleOCR (CPU fallback): {len(ocr_results)} text lines')

for r in ocr_results[:10]:
    if r['type'] == 'text':
        print(f"  [{r['conf']:.2f}] {r['text']}")
    else:
        print(f"  [TABLE] {r['html'][:80]}...")

## 6. Format OCR output for Gemma 4 prompt

We pass the OCR lines as a JSON block alongside the image. Gemma 4 uses both:
- **Vision**: sees the full form image for layout context
- **OCR JSON**: pre-read text (reduces hallucination, especially for diacritics)

In [ ]:
import json

ocr_lines = [
    {'text': r['text'], 'conf': round(r.get('conf', 0.0), 3)}
    for r in ocr_results
    if r['type'] == 'text' and r.get('text', '').strip()
]

ocr_json_str = json.dumps(ocr_lines, ensure_ascii=False, indent=2)
print(f'OCR lines for prompt ({len(ocr_lines)} lines):')
print(ocr_json_str[:800], '...' if len(ocr_json_str) > 800 else '')

SYSTEM_PROMPT = (
    'You are a Vietnamese document understanding assistant.\n'
    'You receive:\n'
    '1. An image of a Vietnamese form or document\n'
    '2. OCR-extracted text lines as JSON (text + confidence)\n'
    '\n'
    'Your task: extract structured data from the document.\n'
    '\n'
    'Return ONLY valid JSON in this exact schema:\n'
    '{\n'
    '  "document_type": "<type of document>",\n'
    '  "fields": {\n'
    '    "<field_name_snake_case>": "<value>"\n'
    '  },\n'
    '  "tables": [\n'
    '    {\n'
    '      "headers": ["col1", "col2"],\n'
    '      "rows": [["val", "val"]]\n'
    '    }\n'
    '  ],\n'
    '  "notes": "<any additional info>"\n'
    '}\n'
    '\n'
    'Rules:\n'
    '- Correct obvious OCR errors using context (Vietnamese diacritics are often misread)\n'
    '- Use low-confidence lines as hints, not gospel\n'
    '- Field names must be in English snake_case\n'
    '- Return raw JSON only — no markdown fences, no explanation'
)

USER_PROMPT = (
    'Extract structured data from this Vietnamese form.\n'
    '\n'
    'OCR-extracted lines:\n'
    + ocr_json_str +
    '\n\nReturn the structured JSON.'
)

print('\nPrompts ready.')

## 7. Load Gemma 4 E4B (4-bit quantized)

`google/gemma-4-e4b-it` is a 4B multimodal model (vision + text). At 4-bit quantization it uses ~4GB VRAM — fits on T4.

> **First run:** ~8GB download, cached to Google Drive for subsequent sessions.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = 'google/gemma-4-e4b-it'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'Device: {DEVICE}  |  VRAM: {props.total_memory / 1e9:.1f} GB  |  GPU: {props.name}')
else:
    print('Device: CPU (inference will be slow)')

bnb_config = None
if DEVICE == 'cuda':
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

print(f'Loading {MODEL_ID} ...')
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map='auto' if DEVICE == 'cuda' else None,
    torch_dtype=torch.bfloat16 if DEVICE == 'cuda' else torch.float32,
)
if DEVICE == 'cpu':
    model = model.to('cpu')

# Set to inference mode (no gradient tracking needed)
_ = model.requires_grad_(False)
print('Gemma 4 E4B loaded (inference-only mode)')
if DEVICE == 'cuda':
    allocated = torch.cuda.memory_allocated() / 1e9
    print(f'VRAM used: {allocated:.2f} GB')

## 8. Structured extraction via Gemma 4

We pass the form image + OCR JSON through Gemma 4's vision+text pipeline and request structured JSON output.

In [ ]:
from PIL import Image

pil_form = Image.open(TEST_IMAGE_PATH).convert('RGB')

messages = [
    {
        'role': 'system',
        'content': [{'type': 'text', 'text': SYSTEM_PROMPT}],
    },
    {
        'role': 'user',
        'content': [
            {'type': 'image', 'image': pil_form},
            {'type': 'text', 'text': USER_PROMPT},
        ],
    },
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors='pt',
)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

print(f'Input tokens: {inputs["input_ids"].shape[1]}')

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=False,
        temperature=1.0,
    )

input_len = inputs['input_ids'].shape[1]
new_tokens = output_ids[0][input_len:]
raw_output = processor.decode(new_tokens, skip_special_tokens=True)
print('\nGemma 4 raw output:')
print(raw_output)

## 9. Parse + validate structured JSON output

In [ ]:
import json, re

def extract_json(text: str) -> dict:
    """Extract JSON from model output, handling markdown fences."""
    text = re.sub(r'^```(?:json)?\s*', '', text.strip(), flags=re.MULTILINE)
    text = re.sub(r'```\s*$', '', text.strip(), flags=re.MULTILINE)
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
        raise ValueError(f'No valid JSON found in output:\n{text[:500]}')

try:
    structured = extract_json(raw_output)
    print('Parsed structured output:')
    print(json.dumps(structured, ensure_ascii=False, indent=2))
except (json.JSONDecodeError, ValueError) as e:
    print(f'JSON parse error: {e}')
    structured = None

## 10. Ground truth comparison — field accuracy

Compare extracted fields against known ground truth from the synthetic form.

In [ ]:
from jiwer import cer

GROUND_TRUTH_FIELDS = {
    'full_name': 'Nguyen Thi Hoa',
    'date_of_birth': '15/03/1985',
    'gender': 'Nu',
    'address': '123 Duong Le Loi, Quan 1, TP.HCM',
    'id_number': '012345678901',
    'phone': '0901234567',
    'insurance_number': 'HS4010012345678',
    'reason': 'Dau dau, sot 3 ngay',
}

if structured and 'fields' in structured:
    extracted = structured['fields']
    print(f'{"Field":<22} {"Extracted":<35} {"Ground Truth":<35} Match')
    print('-' * 100)
    matches, total = 0, 0
    for key, gt_val in GROUND_TRUTH_FIELDS.items():
        pred_val = extracted.get(key, '<missing>')
        exact = pred_val.strip() == gt_val.strip()
        field_cer = cer([gt_val], [pred_val]) if pred_val != '<missing>' else 1.0
        matches += int(exact)
        total += 1
        marker = 'ok' if exact else f'MISS (CER={field_cer:.2f})'
        print(f'{key:<22} {pred_val:<35} {gt_val:<35} {marker}')

    print(f'\nField exact match: {matches}/{total} = {matches/total:.1%}')

    if 'tables' in structured and structured['tables']:
        print(f"\nTables found: {len(structured['tables'])}")
        for i, tbl in enumerate(structured['tables']):
            n_rows = len(tbl.get('rows', []))
            hdrs = tbl.get('headers', [])
            print(f'  Table {i+1}: {n_rows} rows, headers: {hdrs}')
else:
    print('No structured output to compare — check cell 8 for parse errors.')

## 11. Next steps

### Improving accuracy
| Lever | What to try |
|---|---|
| Better OCR | Enable GPU → use `paddlepaddle-gpu` + PP-StructureV3 for layout-aware text regions |
| Prompt engineering | Add few-shot examples of Vietnamese forms in system prompt |
| Thinking mode | Enable `enable_thinking=True` on Gemma 4 for harder fields |
| VietOCR recognizer | Replace PaddleOCR recognizer with VietOCR for Vietnamese-specific accuracy |

### Scaling to real documents
1. Load real scanned forms from HuggingFace or local Drive folder
2. Run the full pipeline in batch with `tqdm`
3. Compute field accuracy, CER, and DER (Diacritic Error Rate from `src/metrics.py`)

### Integration with ViRecognAgent pipeline
- The confidence-gated agent pod in `src/agents.py` can wrap the Gemma 4 call in this notebook
- Low-confidence OCR lines (conf < 0.85) go to Agent Pod for correction
- High-confidence lines pass directly to structured extraction